## Pooling and Deduplication of Results

### Overview

This notebook pools and deduplicates results from all three search engines.
This process hasn't been run yet, and this notebook is meant to function simply as a heuristic, indicating how this process will proceed, once the bad-OCR full-ECCO results are mapped to corresponding passages in the ECCO-TCP.

### 1. Libraries

In [ ]:
import pandas as pd
import re
from collections import defaultdict
from tqdm import tqdm

### 2. Configuration

In [ ]:
# File paths (adapt as needed)
CANONICAL_CHUNKS_CSV = "canonical_ecco_tcp_chunks.csv"  # canon_id, volume_id, text
CLUSTER_RESULTS_CSV  = "cluster_results.csv"            # cluster-based ECCO (bad OCR)
VECTOR_RESULTS_CSV  = "vector_results.csv"              # vector-based ECCO-TCP
HATHI_RESULTS_CSV  = "hathi_results.csv"                # HathiTrust pages

# Similarity threshold for accepting a canonical match
JACCARD_THRESHOLD = 0.5   # this can be tuned up/down

# For speed: limit max canonical candidates per volume to check
# (If volumes are huge, it may be best to pre-filter, but for now, brute force)
MAX_CANONICAL_PER_VOLUME = None  # or an integer like 500 if needed

### 3. Define helper functions

In [ ]:
token_pattern = re.compile(r"\w+", re.UNICODE)

def tokenize_to_set(text: str):
    """
    Tokenize text into a set of lowercased word tokens for Jaccard similarity.
    """
    if not isinstance(text, str):
        return set()
    tokens = token_pattern.findall(text.lower())
    return set(tokens)

def jaccard_similarity(set_a, set_b):
    """
    Compute Jaccard similarity between two token sets.
    """
    if not set_a and not set_b:
        return 0.0
    intersection = set_a & set_b
    union = set_a | set_b
    if not union:
        return 0.0
    return len(intersection) / len(union)

### 4. Load data

In [ ]:
canonical_df = pd.read_csv(CANONICAL_CHUNKS_CSV)

# Expect columns: canon_id, volume_id, text
assert {"canon_id", "volume_id", "text"}.issubset(canonical_df.columns)

cluster_df = pd.read_csv(CLUSTER_RESULTS_CSV)
vector_df = pd.read_csv(VECTOR_RESULTS_CSV)
hathi_df = pd.read_csv(HATHI_RESULTS_CSV)

# Ensure each result DF has: query_id, system_id, local_id, volume_id, text
# If not, rename columns here accordingly

### 5. Pool all results

In [ ]:
all_results = pd.concat([cluster_df, vector_df, hathi_df], ignore_index=True)

# Basic sanity check
required_cols = {"query_id", "system_id", "local_id", "volume_id", "text"}
missing = required_cols - set(all_results.columns)
if missing:
    raise ValueError(f"Missing required columns in pooled results: {missing}")

print(f"Total pooled results: {len(all_results)}")

### 6. Identify volumes shared across systems

In [ ]:
# Count how many distinct systems appear per volume
vol_sys_counts = all_results.groupby("volume_id")["system_id"].nunique()

# Volumes that appear in > 1 system
shared_volumes = vol_sys_counts[vol_sys_counts > 1].index.tolist()

print(f"Volumes appearing in multiple systems: {len(shared_volumes)}")

# Split into:
# - candidates for alignment (shared volumes)
# - unique-volume results (no cross-system duplicates possible)
candidates_df = all_results[all_results["volume_id"].isin(shared_volumes)].copy()
unique_volume_df = all_results[~all_results["volume_id"].isin(shared_volumes)].copy()

print(f"Results in shared volumes: {len(candidates_df)}")
print(f"Results in unique volumes: {len(unique_volume_df)}")

### 7. Precompute tokens for canonical ECCO-TCP chunks, grouped by volume

In [ ]:
print("Tokenizing canonical ECCO-TCP chunks...")

canonical_df["token_set"] = canonical_df["text"].apply(tokenize_to_set)

# Build volume_id -> list of canonical rows (as dicts or small DataFrame)
canon_by_volume = defaultdict(list)

for _, row in canonical_df.iterrows():
    volume_id = row["volume_id"]
    if MAX_CANONICAL_PER_VOLUME is not None and len(canon_by_volume[volume_id]) >= MAX_CANONICAL_PER_VOLUME:
        continue
    canon_by_volume[volume_id].append({
        "canon_id": row["canon_id"],
        "token_set": row["token_set"],
        "text": row["text"]
    })

print(f"Prepared canonical passages for {len(canon_by_volume)} volumes.")

### 8. Tokenize candidate result texts

In [ ]:
print("Tokenizing candidate result passages...")
candidates_df["token_set"] = candidates_df["text"].apply(tokenize_to_set)

### 9. Align each candidate to a canonical chunk in the same volume

In [ ]:
def align_row_to_canonical(row, threshold=JACCARD_THRESHOLD):
    """
    Align a single result row to the best matching canonical chunk
    within the same volume, using Jaccard similarity.
    Returns:
      best_canon_id, best_sim, needs_manual (bool)
    """
    vol_id = row["volume_id"]
    token_set = row["token_set"]

    canon_list = canon_by_volume.get(vol_id, [])
    if not canon_list or not token_set:
        return None, 0.0, True  # nothing to compare or empty text

    best_sim = 0.0
    best_canon_id = None

    for c in canon_list:
        sim = jaccard_similarity(token_set, c["token_set"])
        if sim > best_sim:
            best_sim = sim
            best_canon_id = c["canon_id"]

    if best_sim >= threshold:
        return best_canon_id, best_sim, False
    else:
        return best_canon_id, best_sim, True  # keep candidate but flag for manual review


print("Aligning candidate passages to canonical ECCO-TCP chunks...")
aligned_canon_ids = []
aligned_sims = []
aligned_flags = []

# tqdm provides a progress bar in notebooks
for _, row in tqdm(candidates_df.iterrows(), total=len(candidates_df)):
    canon_id, sim, needs_manual = align_row_to_canonical(row)
    aligned_canon_ids.append(canon_id)
    aligned_sims.append(sim)
    aligned_flags.append(needs_manual)

candidates_df["canon_id"] = aligned_canon_ids
candidates_df["jaccard_sim"] = aligned_sims
candidates_df["needs_manual"] = aligned_flags

### 10. Inspect alignment quality

In [ ]:
print("\nAlignment summary (shared volumes only):")
print(candidates_df["needs_manual"].value_counts(dropna=False))

print("\nBasic statistics on Jaccard similarity:")
print(candidates_df["jaccard_sim"].describe())

# Examples: high-confidence vs low-confidence
print("\nExample high-confidence alignments:")
display(candidates_df[candidates_df["needs_manual"] == False].head(5))

print("\nExample low-confidence alignments (manual inspection needed):")
display(candidates_df[candidates_df["needs_manual"] == True].head(5))

### 11. Combine back with unique-volume results

In [ ]:
# For unique-volume results, you can:
#  - Either leave canon_id empty (they don't need cross-system dedup)
#  - Or, if you wish, run the same alignment function on them later.

unique_volume_df["canon_id"] = None
unique_volume_df["jaccard_sim"] = None
unique_volume_df["needs_manual"] = False  # or True, if you want to inspect them too

# Pooled results with canonical mapping where possible
pooled_with_canonical = pd.concat([candidates_df, unique_volume_df], ignore_index=True)

print(f"\nTotal pooled results with canonical mapping fields: {len(pooled_with_canonical)}")

### 12. (Optional) Deduplicate by query + canonical id

In [ ]:
# If you now want the deduplicated pool for annotation:
# Keep one row per (query_id, canon_id), while retaining info about which systems hit it.

# Filter out rows with no canon_id at all (you might want to keep them separately)
pool_for_annotation = pooled_with_canonical[pooled_with_canonical["canon_id"].notnull()].copy()

# Aggregate which systems/ranks contributed each canonical passage
agg = (pool_for_annotation
       .groupby(["query_id", "canon_id"])
       .agg({
           "system_id": lambda x: list(x),
           "local_id": lambda x: list(x),
           "jaccard_sim": "max",
           "needs_manual": "max"  # True if any contributing row needs manual inspection
       })
       .reset_index())

print(f"\nDeduplicated pool size (query_id, canon_id): {len(agg)}")
display(agg.head())